# Block Gaussian Elimination

Chapter 10: Matrix Block Partitioning · *Mathematics for AI and Machine Learning* by Fuheng Wu

Follow elimination steps on a 2x2 block system and track residuals.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-10/block-elimination/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's default 2x2 system [[A, B], [C, D]] [x1, x2] = [b1, b2]
A, B, C, D = 2.0, 0.5, -0.8, 1.7
b1, b2 = 1.2, 2.3

# ── Eliminate x1, solve the Schur system for x2, then back-substitute ───────
S = D - C * B / A
x2 = (b2 - C / A * b1) / S
x1 = (b1 - B * x2) / A
print("S =", S, " x1 =", x1, " x2 =", x2)
print("residuals:", A * x1 + B * x2 - b1, C * x1 + D * x2 - b2)

# ── Same steps with matrix blocks ───────────────────────────────────────────
n, m = 4, 3
M = torch.randn(n + m, n + m, dtype=torch.float64) + 4 * torch.eye(n + m, dtype=torch.float64)
rhs = torch.randn(n + m, dtype=torch.float64)
A_, B_, C_, D_ = M[:n, :n], M[:n, n:], M[n:, :n], M[n:, n:]
r1, r2 = rhs[:n], rhs[n:]

AinvB = torch.linalg.solve(A_, B_)
Ainvr1 = torch.linalg.solve(A_, r1)
S_ = D_ - C_ @ AinvB                         # only an m x m system remains
y2 = torch.linalg.solve(S_, r2 - C_ @ Ainvr1)
y1 = Ainvr1 - AinvB @ y2
x = torch.cat([y1, y2])
print("\nblock elimination == direct solve:", torch.allclose(x, torch.linalg.solve(M, rhs)))
print("residual ||M x - b|| =", (M @ x - rhs).norm().item())

S = 1.9  x1 = 0.23421052631578948  x2 = 1.463157894736842
residuals: 0.0 0.0

block elimination == direct solve: True
residual ||M x - b|| = 2.220446049250313e-16
